In [1]:
import pandas as pd
import geopandas as gpd
import xarray as xr
import numpy as np
import itertools
from pathlib import Path

In [ ]:
数据集读取、空间匹配、序列日期聚类

In [2]:
def parse_hw(path):
    """读取Hydroweb数据集"""
    lines = open(path).readlines()
    meta = dict(x.split('=') for x in lines[0].split(';') if '=' in x)
    df = pd.DataFrame([l.split(';') for l in lines[1:] if not l.startswith('#') and ';' in l]).iloc[:, [1, 3]]
    df.columns, df['date'], df['wl'] = ['date', 'wl'], pd.to_datetime(df.iloc[:, 0].str.strip(), errors='coerce'), pd.to_numeric(df.iloc[:, 1], errors='coerce')
    return {'lon': float(meta['lon']), 'lat': float(meta['lat']), 'ts': df.dropna()}

def parse_dh(path):
    """读取DAHITI数据集"""
    ds = xr.open_dataset(path)
    df = pd.DataFrame({'date': pd.to_datetime(ds['datetime'].values).normalize(), 'wl': ds['water_level'].values}).dropna()
    return {'lon': float(ds.attrs['longitude']), 'lat': float(ds.attrs['latitude']), 'ts': df}

def map_lakes(data, lakes):
    """数据空间匹配,并以Hylak_id命名"""
    df = pd.DataFrame(data)
    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.lon, df.lat), crs=4326)
    return gpd.sjoin(gdf, lakes[['Hylak_id', 'geometry']], predicate='within').set_index('Hylak_id')


# 根据输入数据列表，动态读取并空间匹配
def prep_and_match_data(lake_gpkg, st_gpkg, file_in, dir_hw, dir_dh, active_sources):
    print(f"正在读取和匹配选中的数据源: {active_sources} ...")
    lakes = gpd.read_file(lake_gpkg).to_crs(4326)
    data_dict = {}
    lake_sets = []

    if 'insitu' in active_sources:
        stations = gpd.read_file(st_gpkg).to_crs(4326)
        st_sjoin = gpd.sjoin(stations, lakes[['Hylak_id', 'geometry']], predicate='within')
        insitu = pd.read_csv(file_in, usecols=['date', 'water_level', 'lake_name']).rename(columns={'water_level':'wl', 'lake_name':'lake'})
        insitu['date'] = pd.to_datetime(insitu['date']).dt.normalize()
        insitu_dict = {row.Hylak_id: insitu[insitu.lake == row.lake_name][['date', 'wl']] for _, row in st_sjoin.iterrows()}
        data_dict['insitu'] = insitu_dict
        lake_sets.append(set(insitu_dict.keys()))

    if 'hydroweb' in active_sources:
        hw_mapped = map_lakes([parse_hw(f) for f in Path(dir_hw).glob('*.txt')], lakes)
        data_dict['hydroweb'] = hw_mapped
        lake_sets.append(set(hw_mapped.index))

    if 'dahiti' in active_sources:
        dh_mapped = map_lakes([parse_dh(f) for f in Path(dir_dh).glob('*.nc')], lakes)
        data_dict['dahiti'] = dh_mapped
        lake_sets.append(set(dh_mapped.index))

    common_lakes = set.intersection(*lake_sets) if lake_sets else set()
    return data_dict, common_lakes


def align_dates_by_tolerance(ts_dict, tolerance_days):
    """将相差在 tolerance_days 以内的日期聚类，并以中位数日期作为统一代表日期"""
    if tolerance_days == 0:
        return ts_dict
    
    #收集该湖泊所有数据源的唯一日期并排序
    all_dates = pd.concat([ts['date'] for ts in ts_dict.values()]).drop_duplicates().sort_values().tolist()
    if not all_dates: 
        return ts_dict
    
    #日期聚类：相邻日期如果在容错窗口内，放入同一个簇
    clusters = []
    curr_cluster = [all_dates[0]]
    for d in all_dates[1:]:
        if (d - curr_cluster[0]).days <= tolerance_days:
            curr_cluster.append(d)
        else:
            clusters.append(curr_cluster)
            curr_cluster = [d]
    clusters.append(curr_cluster)
    
    # 生成映射字典：原始日期 -> 代表日期(该簇的中位数日期)
    date_map = {}
    for c in clusters:
        rep_date = c[len(c)//2] 
        for d in c:
            date_map[d] = rep_date
            
    # 替换各数据源的日期，如果多条数据挤到了同一天，则取平均值
    aligned_dict = {}
    for name, ts in ts_dict.items():
        ts_aligned = ts.copy()
        ts_aligned['date'] = ts_aligned['date'].map(date_map)
        ts_aligned = ts_aligned.groupby('date', as_index=False).mean() 
        aligned_dict[name] = ts_aligned
        
    return aligned_dict

精度验证

In [3]:
def get_metrics(df, c1, c2):
    """样本量, Pearson相关系数, 零均值RMSE"""
    if len(df) < 5: return None
    zm1, zm2 = df[c1] - df[c1].mean(), df[c2] - df[c2].mean()
    return len(df), zm1.corr(zm2), np.sqrt(((zm1 - zm2)**2).mean())

# 动态验证与合并数据
def evaluate_and_merge(common_lakes, data_dict, active_sources, tolerance_days):
    print(f"找到 {len(common_lakes)} 个完全匹配的湖泊。\n{'-'*40}")
    export_data = []
    # 只验证 insitu 与其他数据源
    validation_pairs = [('insitu', src) for src in active_sources if src != 'insitu']
    # ====================
    col_map = {'insitu': 'wl_in', 'hydroweb': 'wl_hw', 'dahiti': 'wl_dh'}

    for lid in common_lakes:
        print(f"【Hylak_id: {lid}】")
        # 提取当前湖泊在选中数据源中的数据
        ts_data = {}
        if 'insitu' in active_sources:
            ts_data['insitu'] = data_dict['insitu'][lid].rename(columns={'wl': 'wl_in'})
        if 'hydroweb' in active_sources:
            ts_data['hydroweb'] = data_dict['hydroweb'].loc[lid, 'ts'].rename(columns={'wl': 'wl_hw'})
        if 'dahiti' in active_sources:
            ts_data['dahiti'] = data_dict['dahiti'].loc[lid, 'ts'].rename(columns={'wl': 'wl_dh'})

        ts_data = align_dates_by_tolerance(ts_data, tolerance_days)

        # 交叉验证计算
        for src1, src2 in validation_pairs:
            # src1 固定为 'insitu'，src2 为其他源
            merged = pd.merge(ts_data[src1], ts_data[src2], on='date')
            res = get_metrics(merged, col_map[src1], col_map[src2])
            if res:
                print(f"  [{src1[:8]:>8} vs {src2[:8]:<8}] 匹配天数:{res[0]:<4} | r:{res[1]:.4f} | RMSE_zm:{res[2]:.3f}m")
        print("-" * 40)

        # 动态外连接合并
        df_export = None
        for src in active_sources:
            if df_export is None:
                df_export = ts_data[src]
            else:
                df_export = pd.merge(df_export, ts_data[src], on='date', how='outer')
        
        export_data.append(df_export.assign(Hylak_id=lid))
        
    return export_data


# 数据存储
def save_export_data(export_data, out_csv):
    if export_data:
        final_df = pd.concat(export_data).sort_values(['Hylak_id', 'date'])
        final_df.to_csv(out_csv, index=False, na_rep='NaN')
        print(f"数据已成功保存至: {out_csv}")
    else:
        print("没有可以保存的数据。")

In [4]:
#更改列表内容以控制验证数据集
ACTIVE_SOURCES = ['insitu', 'hydroweb', 'dahiti'] 
    
# 时间容错窗口
# 设置为 0 代表严格按同一天匹配；设置为 3 代表相差3天内的算作同一次观测
TOLERANCE_DAYS = 3 

DIR_HW = r'D:\Desktop\hma-water\hydroweb\hydroweb\hma_lakes'
DIR_DH = r'D:\Desktop\hma-water\dahiti\dahiti'
FILE_IN = r'D:\Desktop\hma-water\lake_station_lakes_all.csv' 
LAKE_GPKG = r'D:\Desktop\hma-water\HydroLakes_v10_hma.gpkg'
ST_GPKG = r'D:\Desktop\hma-water\lake_station.gpkg'
OUT_CSV = r'D:\Desktop\hma-water\merged_data.csv'

data_dict, common_lakes = prep_and_match_data(
    LAKE_GPKG, ST_GPKG, FILE_IN, DIR_HW, DIR_DH, ACTIVE_SOURCES
)

export_list = evaluate_and_merge(
    common_lakes, data_dict, ACTIVE_SOURCES, TOLERANCE_DAYS
)

save_export_data(export_list, OUT_CSV)

正在读取和匹配选中的数据源: ['insitu', 'hydroweb', 'dahiti'] ...
找到 6 个完全匹配的湖泊。
----------------------------------------
【Hylak_id: 1449】
----------------------------------------
【Hylak_id: 1391】
----------------------------------------
【Hylak_id: 1425】
  [  insitu vs hydroweb] 匹配天数:43   | r:0.8453 | RMSE_zm:0.148m
  [  insitu vs dahiti  ] 匹配天数:35   | r:0.8622 | RMSE_zm:0.120m
----------------------------------------
【Hylak_id: 149】
----------------------------------------
【Hylak_id: 1399】
----------------------------------------
【Hylak_id: 1401】
----------------------------------------
数据已成功保存至: D:\Desktop\hma-water\merged_data.csv
